In [ ]:
import os
import numpy as np
import pandas as pd
from keras.preprocessing import image
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.models import Model

base_model = VGG16(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.get_layer('fc1').output)

In [ ]:
def load_npy_images(base_dir):
    images = []
    filenames = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.npy'):
                img = np.load(os.path.join(root, file))
                images.append(img)
                filenames.append(file)
    return images, filenames

def load_labels(label_csv_files):
    labels_dict = {}
    for label_file in label_csv_files:
        if os.path.exists(label_file):
            df = pd.read_csv(label_file, header=None, names=['index', 'label'])
            for idx, row in df.iterrows():
                filename = f"{str(int(row['index'])).zfill(4)}.npy"
                labels_dict[filename] = row['label']
        else:
            print(f"File {label_file} does not exist")
    return labels_dict

In [ ]:
def preprocess_image(img_array):
    if len(img_array.shape) == 3:
        if img_array.shape[0] == 1:
            img_array = np.repeat(img_array, 3, axis=0)
        elif img_array.shape[0] > 3:
            img_array = img_array[:3]
        img_array = np.transpose(img_array, (1, 2, 0))
    img_array = image.array_to_img(img_array)
    img_array = img_array.resize((224, 224))
    img_array = image.img_to_array(img_array)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img_array

In [ ]:
def extract_features(img_array, model):
    img_data = preprocess_image(img_array)
    features = model.predict(img_data)
    flat_features = features.flatten()
    return flat_features


In [ ]:
train_dir = 'MRNet-v1.0/train'
valid_dir = 'MRNet-v1.0/valid'
label_csv_files = ['MRNet-v1.0/train-acl.csv', 'MRNet-v1.0/valid-acl.csv']

train_images, train_filenames = load_npy_images(train_dir)
valid_images, valid_filenames = load_npy_images(valid_dir)
labels_dict = load_labels(label_csv_files)

features_list = []
labels_list = []
filenames_list = []

def process_images(images, filenames):
    for img, filename in zip(images, filenames):
        features = extract_features(img, model)
        features_list.append(features)
        label = labels_dict.get(filename, 'Unknown')
        labels_list.append(label)
        filenames_list.append(filename)
        print(f"Processed {filename}: label = {label}")

process_images(train_images, train_filenames)
process_images(valid_images, valid_filenames)

features_df = pd.DataFrame(features_list)
features_df.insert(0, 'filename', filenames_list)
features_df.insert(1, 'label', labels_list)

features_df.to_csv('extracted_features_with_labels.csv', index=False)
print("Features and labels saved to extracted_features_with_labels.csv")